In [ ]:
import glob
import os
import imagej
import numpy as np
import pandas as pd
import scyjava as sj
from tqdm.notebook import tqdm

# Java configuration: this will install an extra copy of Java 21, ensuring compatibility. You could also override and use your own Java installation, but this is the easiest way to ensure compatibility with Fiji/ImageJ.
sj.config.set_java_constraints(fetch=True, version='21')
# Just in case anyone runs into issues with memory: change 6g(b) to be as large as needed for movie files
sj.config.add_options('-Xmx6g')


# Update this path to point to your FIJI installation. Interactive allows FIJI to open windows. You can leave out the path to download FIJI: https://py.imagej.net/en/latest/01-Starting-PyImageJ.html
ij = imagej.init(r"C:\Program Files\Fiji.app", mode='headless')

# AFTER fiji is initiated, import classes and functions
from java_imports import *
import track_functions

SLF4J(W): No SLF4J providers were found.
SLF4J(W): Defaulting to no-operation (NOP) logger implementation
SLF4J(W): See https://www.slf4j.org/codes.html#noProviders for further details.
SLF4J(W): Class path contains SLF4J bindings targeting slf4j-api versions 1.7.x or earlier.
SLF4J(W): Ignoring binding found at [jar:file:/opt/FIJI/Fiji/jars/logback-classic-1.2.12.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J(W): See https://www.slf4j.org/codes.html#ignoredBindings for an explanation.


In [ ]:
## This chunk: configuration
# Put movie paths here as a list of strings. This can include things like path/to/files/*.nd2 to get all nd2 files in a folder. Alternatively, you can list individual files.
SEARCHPATHS = [
    r"L:\MIGRATED\Lab_Members\Sam_Steen\Data\260918_PAmCherry-Swi6_varied_20ms\*.nd2"
]

# Configure some basic settings
OPTIMIZE_THRESHOLD = True # If turned on, measures the amount of background noise in each movie and adjusts threshold up or down to compensate. If this is enabled, SPFPA is the key parameter for tracking and THRESHOLD is just a starting guess.
SPFPA=.000045 # The key variable if you enable optimize threshold. "Spots Per Frame Per Area" for the frames before the first photoactivation.
USE_ROI = True # Whether to use ROI masks. If provided, each ROI will be processed separately, and the results will be combined into a single CSV file for each movie. If not provided, the entire movie will be processed as a single ROI.
ROI_EXTENSION = "_488_seg.npy" # Search path for ROIs, e.g. "_488_cp_masks.png". Also accepts seg.npy from cellpose
ROI_GROW = 3 # Grow the ROI by this many pixels in each direction to account for drift or molecules just barely inside cells
MAKE_XML = True # Whether to save XML files that can be opened in the TrackMate GUI. These will be saved in the same folder as the movie with the same name but with .xml extension.
DEBUG=False # Whether to print output to console (some output prints no matter what)

# Setup for processing step 1: detection looks for molecules but doesn't link them
# Step 1a: initial round of detection (aim to get some false positives here)
DETECTOR_SETTINGS = {
    'DO_SUBPIXEL_LOCALIZATION' : True, # Always keep at True
    'RADIUS' : JDouble(5.0), # Note that this value is a diameter in the GUI but a radius here!
    'TARGET_CHANNEL' : JInteger(1),
    'THRESHOLD' : JDouble(12.0), # Quality threshold. I'd recommend using the GUI to choose a good value for this. If optimize threshold is enabled, this is a starting guess.
    'DO_MEDIAN_FILTERING' : False,
}

# Step 1b: filter out low quality detections (aim to remove false positives here)
# See GUI for available filters. Invalid filters result in no tracks being kept, so they're fairly easy to spot
# Note that True means "greater than" and false means "less than"
DETECTOR_FILTERS = [
    # FeatureFilter('QUALITY', 40.0, True),
    # FeatureFilter('MEAN_INTENSITY_CH1', 650.0, True),# Mean intensity filter
]


# Setup for processing step 2: tracking previously detected molecules turns them into tracks
# Steps 2a and 2b:tracking links detected molecules together into tracks. Then it joins track segments by closing gaps between them
# **NOTE** use LAP tracker, not simple LAP tracker
TRACKER_SETTINGS = {
    'MAX_FRAME_GAP': JInteger(3), # For joining segments, how many frames can be skipped. ALWAYS 1 GREATER THAN REAL MAX GAP both here and in the GUI! So set to 3 if you have a 2-long gap
    'ALTERNATIVE_LINKING_COST_FACTOR': JDouble(1.05),
    'LINKING_FEATURE_PENALTIES': {}, 
    'LINKING_MAX_DISTANCE': JDouble(10.0), # max distance in pixels between detections being linked across consecutive frames
    'GAP_CLOSING_MAX_DISTANCE': JDouble(10.0), # Max distance in pixels between segments being linked across frame gaps
    'MERGING_FEATURE_PENALTIES': {}, 
    'SPLITTING_MAX_DISTANCE': JDouble(15.0), # Splitting allows the tracking software to start with a single track and have it split into two at some point. This doesn't make sense with single molecules, of course
    'BLOCKING_VALUE': JDouble(float('inf')),
    'ALLOW_GAP_CLOSING': True, 
    'ALLOW_TRACK_SPLITTING': False, # See above
    'ALLOW_TRACK_MERGING': False, # Opposite of splitting
    'MERGING_MAX_DISTANCE': JDouble(15.0), # See above
    'SPLITTING_FEATURE_PENALTIES': {}, 
    'CUTOFF_PERCENTILE': JDouble(0.9), 
    'GAP_CLOSING_FEATURE_PENALTIES': {}
}

# Step 2c: filter out tracks. Again, see GUI for available filters
TRACKER_FILTERS = [
    FeatureFilter('TRACK_DURATION', 3.5, True), # Requires tracks to be **over** the number given (i.e. 3 frames or more if 2.0 is specified)
    # FeatureFilter('TRACK_DISPLACEMENT', 12.0, False) # Requires tracks to be under 12 pixels in displacement
]

In [ ]:
## This chunk: main processing loop
# java_detector_settings = ij.py.to_java(DETECTOR_SETTINGS)
java_tracker_settings = ij.py.to_java(TRACKER_SETTINGS)
init_detector_quality = DETECTOR_SETTINGS['THRESHOLD']

# Flatten search paths into a single list of files
filenames = [f for s in SEARCHPATHS for f in sorted(glob.glob(s))]

# For each movie
for movie in tqdm(filenames):
    base_filepath, extension = os.path.splitext(movie)
    
    imp = ij.IJ.openImage(movie)

    if OPTIMIZE_THRESHOLD:
        best_qual = track_functions.optimize_qual_threshold(ij, imp, SPFPA, init_detector_quality, DETECTOR_SETTINGS, DEBUG)
        # Set final calibrated quality
        DETECTOR_SETTINGS['THRESHOLD'] = JDouble(best_qual)
        java_detector_settings = ij.py.to_java(DETECTOR_SETTINGS)
    else:
        java_detector_settings = ij.py.to_java(DETECTOR_SETTINGS)


    # Import ROIs and get them from Cellpose output/images to ROI manager objects
    if USE_ROI:
        rois = track_functions.set_rois_from_file(ij, base_filepath, ROI_EXTENSION, ROI_GROW)
    else:
        rois = ["No ROI Provided"]
        
    all_spots_data = []
    global_track_num = 0

    xml_paths = []

    # Now loop through each ROI. Each one is tracked separately, which ensures that molecules can't jump between different ROIs
    for r, roi_obj in enumerate(rois):
        model = Model()
        if DEBUG:
            model.setLogger(Logger.IJ_LOGGER)
        else:
            model.setLogger(Logger.VOID_LOGGER)
        settings = Settings(imp)
        
        if str(roi_obj) != "No ROI Provided":
            settings.setRoi(roi_obj)
            
        settings.detectorFactory = LogDetectorFactory()
        settings.detectorSettings = java_detector_settings
        for filt in DETECTOR_FILTERS:
            settings.addSpotFilter(filt)
            
        settings.trackerFactory = SparseLAPTrackerFactory()
        settings.trackerSettings = java_tracker_settings
        settings.addAllAnalyzers()
        for filt in TRACKER_FILTERS:
            settings.addTrackFilter(filt)

        trackmate = TrackMate(model, settings)
        
        if not trackmate.checkInput() or not trackmate.execDetection():
            print(f"Detection error for ROI {r}: {trackmate.getErrorMessage()}")
            continue 
            
        trackmate.computeSpotFeatures(True)
        trackmate.execSpotFiltering(True)
        
        if model.getSpots().getNSpots(True) == 0:
            continue 
        
        if not trackmate.execTracking():
            print(f"Tracking error for ROI {r}: {trackmate.getErrorMessage()}")
            continue
            
        trackmate.computeTrackFeatures(True)
        trackmate.execTrackFiltering(True)

        if MAKE_XML:
            # Export to XML
            xml_output_file = File(f"{base_filepath}_ROI-{r}_trackmate.xml")
            xml_paths.append(f"{base_filepath}_ROI-{r}_trackmate.xml")
            writer = TmXmlWriter(xml_output_file)
            writer.appendModel(model)
            writer.appendSettings(settings)
            writer.writeToFile()
        
        
        # Extract tracks
        for track_id in model.getTrackModel().trackIDs(True):
            for spot in model.getTrackModel().trackSpots(track_id):
                all_spots_data.append({
                    "frame": spot.getFeature('FRAME'),
                    "x": spot.getFeature('POSITION_X'),
                    "y": spot.getFeature('POSITION_Y'),
                    "track_num": global_track_num,
                    "roi_num": r
                })
            global_track_num += 1
            
        # Free memory references inside the inner loop
        model = settings = trackmate = None
        ij.IJ.run("Collect Garbage")
            
    if all_spots_data:
        pd.DataFrame(all_spots_data).to_csv(f"{base_filepath}_tracks.csv", index=False)
        if MAKE_XML and len(rois) > 1:
            # then we need to merge xml files
            track_functions.combine_trackmate_fov(xml_paths, DEBUG)
    elif DEBUG:
        print(f"No tracks generated for {base_filepath}")

    # Clean up main image memory
    imp.changes = False 
    imp.close()
    ij.IJ.run("Collect Garbage")

print("Processing complete!")

Operating in headless mode - the original ImageJ will have limited functionality.


  0%|          | 0/29 [00:00<?, ?it/s]

Operating in headless mode - the IJ class will not be fully functional.
